# Realistic LLM Use Case: When LLMs Actually Win
**SNIA DSN AI Stack Webinar Series — 2026**

This notebook demonstrates a realistic storage engineering task — classifying **unstructured incident report messages** into 6 categories — where LLM fine-tuning meaningfully outperforms traditional ML.

### Key Finding
TF-IDF + XGBoost achieves ~70–80%. SFT achieves ~85–92%. The gap comes from shared vocabulary across categories that only contextual understanding can disambiguate.

### Dataset
720 synthetic incident reports (120 per category, 6 categories) with deliberately overlapping vocabulary.

## Two Modes

| Mode | What happens | Time |
|------|-------------|------|
| **Full Training** (Step 8A) | Train LoRA adapter from scratch on GPU | ~20 min on T4 |
| **Demo** (Step 8B) | Load pre-trained adapter from HuggingFace Hub | ~2 min |

## Prerequisites
- GPU runtime (T4 or A100) — Go to **Runtime > Change runtime type > T4 GPU**
- HuggingFace token (optional, for Step 8A model upload)

## How to Use
1. Run Steps 1–7 (data generation and baseline)
2. Choose **either** Step 8A (full training) **or** Step 8B (demo mode) — not both
3. Run Steps 9–12 (evaluation and analysis)

## Step 1: Prerequisites and HuggingFace Setup

Install all required packages, check GPU availability, and configure the HuggingFace token.

A HuggingFace token is needed for **Step 8A** to upload the trained model after training completes. If you only plan to use **Step 8B** (demo mode), you can leave the token blank.

To get your token:
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Create a new token with **write** access
3. Paste it in the `HF_TOKEN` variable below

In [ ]:
# Step 1: Prerequisites and HuggingFace Setup
!pip install -q torch transformers datasets accelerate peft trl scikit-learn xgboost matplotlib seaborn sentencepiece huggingface_hub

import torch
import time as _time

_notebook_start = _time.time()

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {mem:.1f} GB")
else:
    print("WARNING: GPU recommended for SFT training.")
    print("Go to Runtime > Change runtime type > T4 GPU")

# HuggingFace token (required for Step 8A model upload)
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = ""  # <-- Paste your token here

# Repository for the pre-trained adapter
HF_REPO = "gidft/smollm2-error-classifier-sft"

# Model size: "360M" (fast, 14 min) or "1.7B" (better accuracy, ~35 min)
MODEL_SIZE = "1.7B"

# Auto-set model name and HF repo based on size
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M" if MODEL_SIZE == "360M" else "HuggingFaceTB/SmolLM2-1.7B"
HF_REPO = f"gidft/smollm2-error-classifier-sft-{MODEL_SIZE.lower()}"

## Step 2: Load Diverse Error Log Dataset

Instead of generating from templates (which creates patterns TF-IDF can exploit), we use a dataset of **714 unique log entries** generated by Claude. Each entry is 2-5 lines of realistic syslog output with:

- **No two entries share the same structure** — every entry is unique
- **Vocabulary is shared across all categories** — timeout, latency, error, failed, degraded appear everywhere
- **Classification requires reading the full narrative** — no single keyword determines the category

The entries were generated by  using the Anthropic API.

In [ ]:
# Step 2: Load Diverse Error Log Dataset
# Generated by scripts/generate_diverse_logs.py using Claude API
# 714 unique, LLM-generated log entries — no templates, no patterns

import json
import urllib.request
import random
import numpy as np

random.seed(42)
np.random.seed(42)

# Load from GitHub (works in Colab without local files)
DATA_URL = "https://raw.githubusercontent.com/provandal/post-training-explorer/main/scripts/diverse_log_entries.json"

print("Loading diverse log entries...")
try:
    with urllib.request.urlopen(DATA_URL) as resp:
        data = json.loads(resp.read().decode())
    print(f"Loaded from GitHub")
except Exception as e:
    # Fallback: try local file
    with open("../scripts/diverse_log_entries.json") as f:
        data = json.load(f)
    print(f"Loaded from local file")

CATEGORIES = data["metadata"]["categories"]

# Build samples list
samples = []
for cat in CATEGORIES:
    entries = data["entries"][cat]
    for text in entries:
        samples.append({"text": text, "label": cat})

random.shuffle(samples)

from collections import Counter
cat_counts = Counter(s["label"] for s in samples)
print(f"Total samples: {len(samples)}")
print(f"Categories: {CATEGORIES}")
for cat in CATEGORIES:
    print(f"  {cat}: {cat_counts[cat]} entries")


## Step 3: Explore the Data

Let's look at 2 examples from each category. Notice how the log lines use shared vocabulary across categories — the same words (timeout, latency, error, degraded, controller, cache) appear everywhere. The classification depends on the **combination and context** of the log lines, not any single keyword.

In [ ]:
# Step 3: Data is already loaded from Step 2
# Just verify the dataset
print(f"Dataset ready: {len(samples)} samples across {len(CATEGORIES)} categories")
print(f"Average log entry length: {np.mean([len(s['text']) for s in samples]):.0f} characters")
print(f"Average lines per entry: {np.mean([s['text'].count(chr(10))+1 for s in samples]):.1f}")

## Step 4: Explore the Data

Display 2 examples per category to see the multi-line log entries. Each sample contains multiple syslog lines separated by newlines. Notice how the same error type can be expressed with completely different vocabulary and how different categories share terminology like "latency," "timeout," "controller," and "firmware."

In [ ]:
# Step 4: Explore the Data
for cat in CATEGORIES:
    cat_samples = [s for s in samples if s['label'] == cat][:2]
    print(f"\n{'='*70}")
    print(f"  {cat}")
    print(f"{'='*70}")
    for i, s in enumerate(cat_samples):
        print(f"\n  Example {i+1}:")
        # Each sample is multi-line (\n-separated log lines)
        for log_line in s['text'].split('\n'):
            print(f"    {log_line}")
print()

## Step 5: Baseline — TF-IDF + XGBoost

First, we establish a traditional ML baseline. TF-IDF converts text to word-frequency features, then XGBoost classifies. This is a strong baseline for text classification — but it treats each message as a bag of words, ignoring word order and context.

In [ ]:
# Step 5: Baseline — TF-IDF + XGBoost
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import time

# Prepare data
texts = [s['text'] for s in samples]
labels = [s['label'] for s in samples]

le = LabelEncoder()
y = le.fit_transform(labels)

# Train/test split
texts_train, texts_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
print("Vectorizing with TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(texts_train)
X_test_tfidf = tfidf.transform(texts_test)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Feature matrix: {X_train_tfidf.shape}")

# Train XGBoost
print("\nTraining XGBoost on TF-IDF features...")
t0 = time.time()
xgb_text = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                          eval_metric='mlogloss')
xgb_text.fit(X_train_tfidf, y_train)
xgb_train_time = time.time() - t0

# Evaluate
xgb_preds = xgb_text.predict(X_test_tfidf)
xgb_accuracy = accuracy_score(y_test, xgb_preds)

print(f"\nTraining time: {xgb_train_time:.2f}s")
print(f"Accuracy: {xgb_accuracy:.1%}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_preds, target_names=le.classes_))

## Step 6: Why XGBoost Struggles Here

The accuracy above is decent but limited. Three common failure modes:

1. **Vocabulary overlap:** "latency," "timeout," "controller," and "firmware" appear in 3+ categories — TF-IDF cannot distinguish context
2. **Negation context:** "No device errors — timeouts caused by space exhaustion" contains drive-failure vocabulary but the meaning is capacity warning
3. **Cross-category causation:** "Cache destage failed to device" sounds like firmware/performance but is actually a drive failure causing the cache issue

An LLM processes the full message as a sequence, understanding context, negation, and causal relationships rather than just word frequency.

## Step 7: Prepare SFT Training Data

Format each sample as a prompt/completion pair for supervised fine-tuning. The prompt includes the full incident report and asks for a classification; the completion is the category label. This is the same format used in the Post-Training Pipeline.

In [ ]:
# Step 7: Prepare SFT Training Data
from datasets import Dataset
from sklearn.model_selection import train_test_split as tts2

def format_for_sft(sample):
    prompt = (
        f"Classify the following storage error log message into one of these categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Log message: {sample['text']}\n\n"
        f"Classification:"
    )
    completion = f" {sample['label']}"
    return {"prompt": prompt, "completion": completion}

all_formatted = [format_for_sft(s) for s in samples]

# Split formatted data the same way as Step 5
random.seed(42)
np.random.seed(42)
fmt_train, fmt_test = tts2(all_formatted, test_size=0.2, random_state=42,
                            stratify=[s['label'] for s in samples])

train_dataset = Dataset.from_dict({
    "prompt": [s["prompt"] for s in fmt_train],
    "completion": [s["completion"] for s in fmt_train],
})

print(f"SFT training samples: {len(train_dataset)}")
print(f"\nExample training input:")
print(f"  Prompt (first 300 chars): {fmt_train[0]['prompt'][:300]}...")
print(f"  Completion: {fmt_train[0]['completion']}")

## Step 8A: Full Training Mode (skip for demo)

Train a LoRA adapter on SmolLM2-360M from scratch. This takes ~20 minutes on a T4 GPU. After training, the adapter is saved locally and automatically uploaded to the HuggingFace Hub if `HF_TOKEN` was set in Step 1.

**Skip this cell** if you want a quick demo — use Step 8B instead to load a pre-trained adapter.

In [ ]:
# Step 8A: Full Training Mode (~20 min on T4)
# SKIP THIS CELL for demo mode — use Step 8B instead

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for training. Go to Runtime > Change runtime type > T4 GPU")

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

set_seed(42)

# MODEL_NAME set in Step 1 based on MODEL_SIZE
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA configuration — same settings as Post-Training Pipeline
lora_rank = 32 if MODEL_SIZE == "360M" else 64
lora_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_rank * 2,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Training configuration
training_args = SFTConfig(
    output_dir="/tmp/sft_error_logs",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("\nStarting SFT training...")
t0 = time.time()
trainer.train()
sft_train_time = time.time() - t0
print(f"\nTraining completed in {sft_train_time/60:.1f} minutes")

# Save adapter locally
model.save_pretrained("/tmp/sft_error_logs/final_adapter")
tokenizer.save_pretrained("/tmp/sft_error_logs/final_adapter")
print("Adapter saved to /tmp/sft_error_logs/final_adapter")

# Upload to HuggingFace Hub
if HF_TOKEN:
    import os
    cache_path = os.path.expanduser("~/.cache/huggingface/token")
    if os.path.exists(cache_path):
        os.remove(cache_path)
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Uploaded to {HF_REPO}")
else:
    print("No HF_TOKEN set in Step 1 — skipping upload.")

## Step 8B: Demo Mode — Load Pre-trained Adapter

Load a pre-trained LoRA adapter from the HuggingFace Hub. This skips the ~20 minute training step and lets you jump straight to evaluation. The adapter was trained with the exact same configuration as Step 8A.

In [ ]:
# Step 8B: Demo Mode — Load Pre-trained Adapter from HuggingFace Hub
# USE THIS CELL instead of Step 8A for a quick demo

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftModel

set_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for inference. Go to Runtime > Change runtime type > T4 GPU")

# MODEL_NAME and HF_REPO set in Step 1 based on MODEL_SIZE
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading base model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading pre-trained adapter: {HF_REPO}...")
model = PeftModel.from_pretrained(base_model, HF_REPO)
print("Adapter loaded successfully.")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {trainable:,} trainable / {total:,} total")

# Mark that we skipped training
sft_train_time = 0

## Step 9: Evaluate the SFT Model

Run inference on the test set and compare predictions against ground truth. Each sample is formatted with the same prompt template used during training.

In [ ]:
# Step 9: Evaluate the SFT Model
model.eval()

# Reconstruct test set
_, test_samples_eval = tts2(samples, test_size=0.2, random_state=42,
                             stratify=[s['label'] for s in samples])

sft_preds = []
sft_correct = 0
total_test = len(test_samples_eval)

print(f"Evaluating SFT model on {total_test} test samples...")
for idx, sample in enumerate(test_samples_eval):
    if (idx + 1) % 20 == 0 or idx == 0:
        print(f"  Progress: {idx+1}/{total_test}")

    prompt = (
        f"Classify the following storage error log message into one of these categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Log message: {sample['text']}\n\n"
        f"Classification:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # Check if any category name appears in the output
    predicted = None
    for cat in CATEGORIES:
        if cat.lower() in generated.lower():
            predicted = cat
            break

    is_correct = predicted == sample['label']
    if is_correct:
        sft_correct += 1
    sft_preds.append({
        'true': sample['label'],
        'predicted': predicted or generated[:50],
        'raw_output': generated[:100],
        'correct': is_correct,
    })

sft_accuracy = sft_correct / len(test_samples_eval)
print(f"\nSFT Accuracy: {sft_accuracy:.1%} ({sft_correct}/{len(test_samples_eval)})")

# Per-category breakdown
print(f"\nPer-Category Results:")
print("-" * 50)
for cat in CATEGORIES:
    cat_preds = [p for p in sft_preds if p['true'] == cat]
    cat_correct = sum(1 for p in cat_preds if p['correct'])
    cat_total = len(cat_preds)
    pct = cat_correct / cat_total if cat_total > 0 else 0
    bar = "#" * int(pct * 10) + "." * (10 - int(pct * 10))
    print(f"  {cat:<25s} {bar} {cat_correct}/{cat_total} ({pct:.0%})")

## Step 10: Head-to-Head Comparison

Side-by-side charts comparing TF-IDF + XGBoost vs SFT fine-tuning on overall accuracy and per-category accuracy, plus a summary table with training time, model size, and qualitative differences.

In [ ]:
# Step 10: Head-to-Head Comparison
import matplotlib.pyplot as plt
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy comparison
models = ['TF-IDF + XGBoost', f'SFT (SmolLM2-{MODEL_SIZE})']
accs = [xgb_accuracy, sft_accuracy]
colors = ['#f97316', '#3b82f6']
bars = ax1.bar(models, accs, color=colors, alpha=0.8, width=0.5)
ax1.set_ylabel('Accuracy')
ax1.set_title('Text Classification Accuracy')
ax1.set_ylim(0, 1.1)
for bar, val in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.1%}', ha='center', fontweight='bold', fontsize=13)

# Right: Per-category comparison
xgb_cat_acc = {}
sft_cat_acc = {}
for cat in CATEGORIES:
    # XGBoost
    cat_mask = [le.classes_[y_test[i]] == cat for i in range(len(y_test))]
    cat_indices = [i for i, m in enumerate(cat_mask) if m]
    if cat_indices:
        xgb_cat_acc[cat] = sum(1 for i in cat_indices if xgb_preds[i] == y_test[i]) / len(cat_indices)
    # SFT
    cat_sft = [p for p in sft_preds if p['true'] == cat]
    if cat_sft:
        sft_cat_acc[cat] = sum(1 for p in cat_sft if p['correct']) / len(cat_sft)

x = np.arange(len(CATEGORIES))
width = 0.35
ax2.bar(x - width/2, [xgb_cat_acc.get(c, 0) for c in CATEGORIES], width,
        label='TF-IDF + XGBoost', color='#f97316', alpha=0.8)
ax2.bar(x + width/2, [sft_cat_acc.get(c, 0) for c in CATEGORIES], width,
        label='SFT', color='#3b82f6', alpha=0.8)
ax2.set_ylabel('Accuracy')
ax2.set_title('Per-Category Accuracy')
ax2.set_xticks(x)
ax2.set_xticklabels(CATEGORIES, rotation=35, ha='right', fontsize=8)
ax2.legend()
ax2.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 70)
print("  COMPARISON: TF-IDF + XGBoost vs SFT Fine-Tuning")
print("=" * 70)
if sft_train_time > 0:
    sft_time_str = f'{sft_train_time/60:.1f} min'
else:
    sft_time_str = 'Pre-trained (skipped)'

comparison_data = {
    'Metric': ['Accuracy', 'Training Time', 'Model Size', 'Hardware', 'Handles Novel Phrasing', 'Contextual Understanding'],
    'TF-IDF + XGBoost': [f'{xgb_accuracy:.1%}', f'{xgb_train_time:.1f}s', '~5 MB', 'CPU', 'Limited', 'None (bag of words)'],
    'SFT (SmolLM2-{MODEL_SIZE})': [f'{sft_accuracy:.1%}', sft_time_str, '~700 MB' if MODEL_SIZE == '360M' else '~3.4 GB', 'GPU', 'Good', 'Full sequence modeling'],
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("=" * 70)

## Step 11: Error Analysis

Examine cases where the SFT model succeeds and TF-IDF + XGBoost fails. This highlights the LLM's advantage in understanding contextual meaning.

In [ ]:
# Step 11: Error Analysis — Where does the LLM win?
print("Cases where SFT succeeds and TF-IDF + XGBoost fails:")
print("=" * 70)

# Align test samples between the two models
sft_test = test_samples_eval
count = 0
for i, (sample, sft_pred) in enumerate(zip(sft_test, sft_preds)):
    if i >= len(y_test):
        break
    xgb_pred_label = le.classes_[xgb_preds[i]] if i < len(xgb_preds) else None
    xgb_wrong = xgb_pred_label != sample['label']
    sft_right = sft_pred['correct']
    
    if xgb_wrong and sft_right and count < 4:
        count += 1
        print(f"\n  Example {count}:")
        print(f"  True label:    {sample['label']}")
        print(f"  XGBoost pred:  {xgb_pred_label} X")
        print(f"  SFT pred:      {sft_pred['predicted']} OK")
        text = sample['text'][:150]
        print(f"  Log message:   {text}...")

if count == 0:
    print("  (No clear examples in this run -- both models may have similar error patterns)")

print(f"\n\nSummary: SFT's advantage comes from understanding the *meaning* of incident reports,")
print(f"not just keyword frequency. This matters most when:")
print(f"  - Different categories share vocabulary (latency, timeout, controller, firmware)")
print(f"  - Negation changes meaning (no device errors, not a firmware defect)")
print(f"  - Cross-category causation requires reasoning (cache failed because drive failed)")

## Step 12: Conclusion — A Framework for Choosing Your Approach

| Characteristic | Use Traditional ML | Use LLM Fine-Tuning |
|---|---|---|
| **Input format** | Structured numbers, tables | Unstructured text, natural language |
| **Feature engineering** | Clear, known features | Features hard to extract manually |
| **Vocabulary** | N/A or controlled | Variable, natural language |
| **Context dependency** | Low — features are independent | High — meaning depends on context |
| **Training resources** | CPU, seconds-minutes | GPU, minutes-hours |
| **Model size** | KBs-MBs | Hundreds of MBs |
| **Interpretability** | High (feature importance) | Low (black box) |

**The two notebooks together give you a decision framework:**
- **Structured numeric data** (I/O metrics, sensor readings, performance counters) -> Random Forest / XGBoost
- **Unstructured text** (logs, error messages, incident reports, documentation) -> LLM fine-tuning

**Neither approach is universally better.** The right choice depends on your data, not on what's trending.

Return to the [Post-Training Pipeline](Post_Training_Pipeline.ipynb) to run the full SFT -> DPO -> GRPO pipeline, or see the [Traditional ML Comparison](Traditional_ML_Comparison.ipynb) for the structured data perspective.